In [0]:
# Transaction Summary v2 - Testing GitHub Action auto deployment
# Databricks notebook source
# MAGIC %md
# MAGIC # Transaction Summary Analytics

# COMMAND ----------

# Get parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Running for catalog: {catalog}, schema: {schema}")

# COMMAND ----------

# Create schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# COMMAND ----------

# Read sales transaction data
df = spark.sql("""
    SELECT
        o_orderstatus as order_status,
        o_orderpriority as order_priority,
        COUNT(*) as total_orders,
        SUM(o_totalprice) as total_revenue,
        AVG(o_totalprice) as avg_order_value,
        MIN(o_totalprice) as min_order_value,
        MAX(o_totalprice) as max_order_value
    FROM samples.tpch.orders
    GROUP BY
        o_orderstatus,
        o_orderpriority
    ORDER BY
        total_revenue DESC
""")

display(df)

# COMMAND ----------

# Write to target table
df.write \
  .mode("overwrite") \
  .saveAsTable(f"{catalog}.{schema}.transaction_summary")

print(f"Table {catalog}.{schema}.transaction_summary created successfully!")